# Phase 2-3: PatchCore and PaDiM on MVTec AD Pill

Run on a Colab GPU, top to bottom.

Settings are fixed before the test set is used: 256x256 images, WideResNet50-2, anomalib
defaults except PaDiM n_features=300 (550 runs out of memory on a T4), threshold = 99th
percentile of validation image scores.

## 1. Save project code

In [ ]:
%%writefile prepare_dataset.py
"""Validate the MVTec AD Pill folder and write a reproducible train/validation split.

Uses only the standard library so it runs anywhere, including Colab.

    python src/prepare_dataset.py --root /path/to/mvtec --out splits/pill_seed42.json
"""
import argparse
import json
import random
import sys
from pathlib import Path

EXPECTED_TRAIN_GOOD = 267
EXPECTED_TEST_GOOD = 26
EXPECTED_TEST_DEFECT = 141
EXPECTED_DEFECTS = {
    "color",
    "combined",
    "contamination",
    "crack",
    "faulty_imprint",
    "pill_type",
    "scratch",
}


class DatasetError(Exception):
    """The dataset folder does not match the expected structure."""


def find_category_root(root, category="pill"):
    """Return the category folder, whether root is that folder or a parent of it."""
    root = Path(root)
    if (root / "train").is_dir() and (root / "test").is_dir():
        return root
    for candidate in sorted(root.rglob(category)):
        if (candidate / "train").is_dir() and (candidate / "test").is_dir():
            return candidate
    raise DatasetError(f"No '{category}' folder with train/ and test/ under {root}")


def _pngs(folder):
    return sorted(p.name for p in folder.glob("*.png"))


def inspect_dataset(pill_root):
    """Count images and check every defect image has a mask. Raises DatasetError."""
    pill_root = Path(pill_root)
    problems = []

    train_good = _pngs(pill_root / "train" / "good")
    test_good = _pngs(pill_root / "test" / "good")
    test_dirs = {p.name for p in (pill_root / "test").iterdir() if p.is_dir()}
    defects = sorted(test_dirs - {"good"})

    if set(defects) != EXPECTED_DEFECTS:
        problems.append(
            f"defect folders differ: missing {sorted(EXPECTED_DEFECTS - set(defects))}, "
            f"unexpected {sorted(set(defects) - EXPECTED_DEFECTS)}"
        )

    defect_counts = {}
    for defect in defects:
        images = _pngs(pill_root / "test" / defect)
        defect_counts[defect] = len(images)
        mask_dir = pill_root / "ground_truth" / defect
        masks = set(_pngs(mask_dir)) if mask_dir.is_dir() else set()
        expected_masks = {f"{Path(name).stem}_mask.png" for name in images}
        missing = sorted(expected_masks - masks)
        if missing:
            problems.append(f"{defect}: {len(missing)} missing masks, e.g. {missing[0]}")

    total_defect = sum(defect_counts.values())
    for label, actual, expected in (
        ("train/good", len(train_good), EXPECTED_TRAIN_GOOD),
        ("test/good", len(test_good), EXPECTED_TEST_GOOD),
        ("test defects", total_defect, EXPECTED_TEST_DEFECT),
    ):
        if actual != expected:
            problems.append(f"{label}: expected {expected} images, found {actual}")

    if problems:
        raise DatasetError("; ".join(problems))

    return {
        "train_good": len(train_good),
        "test_good": len(test_good),
        "test_defect": total_defect,
        "defect_counts": defect_counts,
    }


def make_split(pill_root, val_ratio=0.15, seed=42):
    """Split train/good only. The test set is never touched."""
    if not 0 < val_ratio < 1:
        raise ValueError("val_ratio must be between 0 and 1")
    names = _pngs(Path(pill_root) / "train" / "good")
    random.Random(seed).shuffle(names)
    n_val = round(len(names) * val_ratio)
    return {
        "seed": seed,
        "val_ratio": val_ratio,
        "train": sorted(names[n_val:]),
        "validation": sorted(names[:n_val]),
    }


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--root", required=True, help="pill folder or a parent of it")
    parser.add_argument("--out", default="splits/pill_seed42.json")
    parser.add_argument("--val-ratio", type=float, default=0.15)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args(argv)

    try:
        pill_root = find_category_root(args.root)
        summary = inspect_dataset(pill_root)
    except DatasetError as error:
        print(f"Dataset check failed: {error}", file=sys.stderr)
        return 1

    split = make_split(pill_root, args.val_ratio, args.seed)
    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(split, indent=2) + "\n")

    print(json.dumps(summary, indent=2))
    print(f"train={len(split['train'])} validation={len(split['validation'])} -> {out}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile ad_metrics.py
"""Evaluation metrics for anomaly detection. Model independent."""
import numpy as np
from scipy import ndimage
from sklearn.metrics import roc_auc_score


def pick_threshold(validation_scores, quantile=0.99):
    """Image-score threshold taken from normal validation images only."""
    return float(np.quantile(np.asarray(validation_scores, dtype=np.float64), quantile))


def au_pro(masks, maps, fpr_limit=0.3, steps=200):
    """Area under the per-region-overlap curve up to fpr_limit, scaled to 0-1.

    masks: bool array (N, H, W). maps: float array (N, H, W).
    Each connected defect region counts equally, however large it is.
    """
    masks = np.asarray(masks, dtype=bool)
    maps = np.asarray(maps, dtype=np.float32)
    if masks.shape != maps.shape:
        raise ValueError("masks and maps must have the same shape")
    if not masks.any():
        raise ValueError("au_pro needs at least one defect pixel")

    fprs = np.linspace(0.0, fpr_limit, steps + 1)
    thresholds = np.quantile(maps[~masks], 1.0 - fprs)

    overlaps = []
    for mask, score_map in zip(masks, maps):
        labels, count = ndimage.label(mask)
        for region in range(1, count + 1):
            scores = np.sort(score_map[labels == region])
            below = np.searchsorted(scores, thresholds, side="right")
            overlaps.append(1.0 - below / scores.size)
    pro = np.mean(overlaps, axis=0)

    area = np.sum((pro[1:] + pro[:-1]) / 2 * np.diff(fprs))
    return float(area / fpr_limit)


def evaluate(labels, scores, masks, maps, threshold):
    """Image and pixel metrics for one model. labels: 1 means defective."""
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=np.float64)
    predicted = scores > threshold

    true_positive = int(np.sum(predicted & (labels == 1)))
    false_positive = int(np.sum(predicted & (labels == 0)))
    false_negative = int(np.sum(~predicted & (labels == 1)))
    true_negative = int(np.sum(~predicted & (labels == 0)))

    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    masks = np.asarray(masks, dtype=bool)
    return {
        "image_auroc": float(roc_auc_score(labels, scores)),
        "pixel_auroc": float(roc_auc_score(masks.ravel(), np.asarray(maps).ravel())),
        "au_pro": au_pro(masks, maps),
        "threshold": float(threshold),
        "precision": precision,
        "recall": recall,
        "f1": 2 * precision * recall / max(precision + recall, 1e-12),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
    }


def evaluate_by_defect(groups, scores, masks, maps, threshold):
    """Per-defect results. Each defect is compared against the good images."""
    groups = np.asarray(groups)
    scores = np.asarray(scores, dtype=np.float64)
    masks = np.asarray(masks, dtype=bool)
    maps = np.asarray(maps)
    good = groups == "good"

    rows = {}
    for defect in sorted(set(groups.tolist()) - {"good"}):
        selected = groups == defect
        both = selected | good
        rows[defect] = {
            "images": int(selected.sum()),
            "image_auroc": float(roc_auc_score(selected[both], scores[both])),
            "pixel_auroc": float(roc_auc_score(masks[both].ravel(), maps[both].ravel())),
            "au_pro": au_pro(masks[both], maps[both]),
            "recall": float(np.mean(scores[selected] > threshold)),
        }
    return rows

## 2. Install, load data, apply the committed split

In [ ]:
%pip install -q anomalib kagglehub

In [ ]:
import gc, json, os, shutil, time, warnings
from pathlib import Path

import kagglehub
import numpy as np
import torch
from PIL import Image
from torchvision.transforms.v2 import Compose, Normalize, Resize
from torchvision.transforms.v2.functional import pil_to_tensor, to_dtype

import ad_metrics
import prepare_dataset

warnings.filterwarnings("ignore")
SPLIT = json.loads('''{"seed": 42, "val_ratio": 0.15, "train": ["000.png", "001.png", "002.png", "004.png", "006.png", "007.png", "008.png", "010.png", "011.png", "012.png", "013.png", "014.png", "015.png", "016.png", "017.png", "018.png", "019.png", "020.png", "021.png", "022.png", "023.png", "024.png", "025.png", "026.png", "027.png", "028.png", "030.png", "031.png", "032.png", "033.png", "034.png", "035.png", "036.png", "037.png", "038.png", "039.png", "040.png", "041.png", "042.png", "043.png", "044.png", "045.png", "046.png", "047.png", "048.png", "049.png", "050.png", "052.png", "053.png", "054.png", "055.png", "056.png", "057.png", "058.png", "059.png", "060.png", "061.png", "062.png", "063.png", "064.png", "067.png", "068.png", "069.png", "070.png", "071.png", "073.png", "074.png", "075.png", "076.png", "077.png", "079.png", "080.png", "081.png", "082.png", "083.png", "084.png", "086.png", "087.png", "088.png", "089.png", "090.png", "091.png", "092.png", "093.png", "094.png", "095.png", "096.png", "097.png", "098.png", "099.png", "100.png", "101.png", "102.png", "104.png", "105.png", "107.png", "108.png", "109.png", "110.png", "111.png", "113.png", "114.png", "115.png", "116.png", "117.png", "118.png", "120.png", "121.png", "124.png", "125.png", "126.png", "127.png", "128.png", "129.png", "130.png", "132.png", "133.png", "134.png", "135.png", "136.png", "137.png", "139.png", "140.png", "141.png", "142.png", "143.png", "145.png", "146.png", "147.png", "148.png", "150.png", "151.png", "152.png", "154.png", "155.png", "156.png", "157.png", "158.png", "159.png", "160.png", "161.png", "162.png", "163.png", "164.png", "165.png", "166.png", "167.png", "168.png", "169.png", "170.png", "171.png", "172.png", "173.png", "174.png", "175.png", "176.png", "178.png", "179.png", "180.png", "181.png", "182.png", "183.png", "185.png", "186.png", "187.png", "188.png", "189.png", "190.png", "191.png", "192.png", "193.png", "194.png", "195.png", "196.png", "197.png", "200.png", "202.png", "203.png", "204.png", "205.png", "206.png", "207.png", "208.png", "209.png", "211.png", "212.png", "213.png", "214.png", "216.png", "217.png", "218.png", "219.png", "220.png", "222.png", "223.png", "226.png", "227.png", "228.png", "229.png", "230.png", "231.png", "232.png", "233.png", "234.png", "235.png", "236.png", "237.png", "238.png", "239.png", "240.png", "242.png", "243.png", "245.png", "249.png", "251.png", "252.png", "253.png", "254.png", "255.png", "256.png", "257.png", "258.png", "259.png", "261.png", "262.png", "264.png", "265.png"], "validation": ["003.png", "005.png", "009.png", "029.png", "051.png", "065.png", "066.png", "072.png", "078.png", "085.png", "103.png", "106.png", "112.png", "119.png", "122.png", "123.png", "131.png", "138.png", "144.png", "149.png", "153.png", "177.png", "184.png", "198.png", "199.png", "201.png", "210.png", "215.png", "221.png", "224.png", "225.png", "241.png", "244.png", "246.png", "247.png", "248.png", "250.png", "260.png", "263.png", "266.png"]}''')
PILL = prepare_dataset.find_category_root(kagglehub.dataset_download("ipythonx/mvtec-ad"))
print(prepare_dataset.inspect_dataset(PILL))

WORK = Path("/content/work")
shutil.rmtree(WORK, ignore_errors=True)
for name, files in (("train_good", SPLIT["train"]), ("val_good", SPLIT["validation"])):
    (WORK / name).mkdir(parents=True)
    for file in files:
        os.symlink(PILL / "train" / "good" / file, WORK / name / file)
print("train", len(list((WORK / "train_good").iterdir())), "val", len(list((WORK / "val_good").iterdir())))

## 3. Train

In [ ]:
from anomalib.data import Folder
from anomalib.engine import Engine
from anomalib.models import Padim, Patchcore
from anomalib.pre_processing import PreProcessor
from lightning.pytorch import seed_everything

SIZE = 256
DEVICE = "cuda"
BACKBONE = "wide_resnet50_2"

# Batched matrix inverse crashes CUDA on the T4, so PaDiM inverts on the CPU.
_inverse = torch.linalg.inv


def cpu_inverse(matrices):
    return _inverse(matrices.cpu()).to(matrices.device)


torch.linalg.inv = cpu_inverse


def make_transform(size):
    return Compose([
        Resize((size, size), antialias=True),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def build(name, size):
    common = dict(
        pre_processor=PreProcessor(transform=make_transform(size)),
        post_processor=False, evaluator=False, visualizer=False,
    )
    if name == "patchcore":
        return Patchcore(backbone=BACKBONE, layers=("layer2", "layer3"), coreset_sampling_ratio=0.1, **common)
    return Padim(backbone=BACKBONE, layers=["layer1", "layer2", "layer3"], n_features=300, **common)


def train(name, size=SIZE):
    seed_everything(42)
    datamodule = Folder(
        name="pill", root=WORK, normal_dir="train_good", normal_test_dir="val_good",
        train_batch_size=8, eval_batch_size=8, num_workers=2,
        test_split_mode="from_dir", val_split_mode="same_as_test",
    )
    model = build(name, size)
    engine = Engine(default_root_dir=f"/content/runs/{name}", logger=False, accelerator="gpu", devices=1,
                    enable_progress_bar=False, num_sanity_val_steps=0)
    start = time.perf_counter()
    engine.fit(model=model, datamodule=datamodule)
    print("RESULT", name, "fit seconds", round(time.perf_counter() - start, 1))
    return model.to(DEVICE).eval()

In [ ]:
def load_image(path):
    return to_dtype(pil_to_tensor(Image.open(path).convert("RGB")), torch.float32, scale=True)


@torch.inference_mode()
def predict(model, paths, size=SIZE, batch_size=8):
    transform = make_transform(size)
    scores, maps = [], []
    for start in range(0, len(paths), batch_size):
        batch = torch.stack([transform(load_image(p)) for p in paths[start:start + batch_size]]).to(DEVICE)
        output = model.model(batch)
        scores.append(output.pred_score.flatten().float().cpu().numpy())
        maps.append(output.anomaly_map.squeeze(1).float().cpu().numpy())
    return np.concatenate(scores), np.concatenate(maps)


@torch.inference_mode()
def latency_ms(model, paths, size=SIZE):
    transform = make_transform(size)
    times = []
    for index, path in enumerate(paths):
        torch.cuda.synchronize()
        start = time.perf_counter()
        model.model(transform(load_image(path)).unsqueeze(0).to(DEVICE))
        torch.cuda.synchronize()
        if index >= 5:
            times.append((time.perf_counter() - start) * 1000)
    return float(np.mean(times))


VAL_PATHS = [PILL / "train" / "good" / f for f in SPLIT["validation"]]
TEST_PATHS = sorted((PILL / "test").glob("*/*.png"))
GROUPS = [p.parent.name for p in TEST_PATHS]
LABELS = [int(g != "good") for g in GROUPS]


def load_mask(path, size=SIZE):
    if path.parent.name == "good":
        return np.zeros((size, size), dtype=bool)
    mask_path = PILL / "ground_truth" / path.parent.name / f"{path.stem}_mask.png"
    return np.asarray(Image.open(mask_path).convert("L").resize((size, size), Image.NEAREST)) > 0


MASKS = np.stack([load_mask(p) for p in TEST_PATHS])
print(len(TEST_PATHS), MASKS.shape, sum(LABELS))

## 4. Validation threshold, then final test

In [ ]:
RESULTS, MAPS, SCORES = {}, {}, {}
for name in ("patchcore", "padim"):
    model = train(name)
    val_scores, _ = predict(model, VAL_PATHS)
    threshold = ad_metrics.pick_threshold(val_scores)
    scores, maps = predict(model, TEST_PATHS)
    result = ad_metrics.evaluate(LABELS, scores, MASKS, maps, threshold)
    result["latency_ms"] = latency_ms(model, VAL_PATHS)
    RESULTS[name] = {
        "overall": result,
        "by_defect": ad_metrics.evaluate_by_defect(GROUPS, scores, MASKS, maps, threshold),
        "validation_scores": val_scores.tolist(),
        "test_scores": {f"{g}/{p.name}": float(s) for g, p, s in zip(GROUPS, TEST_PATHS, scores)},
    }
    SCORES[name], MAPS[name] = scores, maps
    print("RESULT", name, json.dumps({k: round(v, 4) for k, v in result.items()}))
    for defect, row in RESULTS[name]["by_defect"].items():
        print("RESULT", name, defect, json.dumps({k: round(v, 4) for k, v in row.items()}))
    del model
    gc.collect()
    torch.cuda.empty_cache()

## 5. Errors and example heatmaps

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

OUT = Path("/content/results")
OUT.mkdir(exist_ok=True)
for name in RESULTS:
    threshold = RESULTS[name]["overall"]["threshold"]
    flagged = SCORES[name] > threshold
    labels = np.array(LABELS)
    print("RESULT", name, "false positives", [f"{g}/{p.name}" for g, p, f, l in zip(GROUPS, TEST_PATHS, flagged, labels) if f and not l])
    print("RESULT", name, "false negatives", [f"{g}/{p.name}" for g, p, f, l in zip(GROUPS, TEST_PATHS, flagged, labels) if l and not f])

    defects = sorted(set(GROUPS) - {"good"})
    fig, axes = plt.subplots(3, len(defects), figsize=(2.4 * len(defects), 7.4))
    low, high = np.quantile(MAPS[name], [0.01, 0.999])
    for column, defect in enumerate(defects):
        index = GROUPS.index(defect)
        image = Image.open(TEST_PATHS[index]).convert("RGB").resize((SIZE, SIZE))
        axes[0, column].imshow(image)
        axes[0, column].set_title(defect, fontsize=9)
        axes[1, column].imshow(MASKS[index], cmap="gray")
        axes[2, column].imshow(image)
        axes[2, column].imshow(MAPS[name][index], cmap="jet", alpha=0.5, vmin=low, vmax=high)
    for axis in axes.ravel():
        axis.axis("off")
    fig.suptitle(name)
    fig.tight_layout()
    fig.savefig(OUT / f"examples_{name}.png", dpi=110)
    plt.show()

(OUT / "phase3_results.json").write_text(json.dumps(RESULTS, indent=1))

## 6. Save results

In [ ]:
from google.colab import files

shutil.make_archive("/content/phase3_results", "zip", OUT)
files.download("/content/phase3_results.zip")